# 07. Multi-Agent RL QMIX Mappo CFR and Agentic Reasoning Loops

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Tan (1993 Independent Q-Learning); Sunehag et al. (2017 Value-Decomposition Networks); Rashid et al. (2018 QMIX); Yu et al. (2022 Multi-Agent PPO / MAPPO); Zinkevich et al. (2007 Counterfactual Regret Minimization); Brown & Sandholm (2019 Pluribus); Du et al. / Liang et al. (2023 Multi-Agent Debate & Deliberation).
- **Purpose:** First-principles mathematical implementation and empirical verification of multi-agent reinforcement learning, imperfect-information game solving, and collaborative agentic loops: QMIX monotonic hypernetwork mixing, MAPPO centralized Generalized Advantage Estimation (GAE), Counterfactual Regret Minimization (CFR) regret matching, and multi-agent reasoning consensus metrics.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. Hierarchical RL Options Feudal Her and POMDP Active Inference](06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.ipynb) $\to$ `🔗 Downstream Syntheses:` [08. Successor Representations Forward Backward Zero Shot RL and DIAYN](08_successor_representations_forward_backward_zero_shot_rl_and_diayn.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never use standard QMIX on non-monotonic payoff environments where coordination demands sub-optimal individual actions (e.g. the Matrix Climb or Penalty game); the monotonicity constraint $\frac{\partial Q_{\text{tot}}}{\partial Q_i} \ge 0$ structurally eliminates non-monotonic Pareto optima. Never run vanilla CFR in games with $N > 2$ players without iteration discounting (DCFR), as standard regret accumulation can oscillate indefinitely.

## Mathematical Foundations & Governing Equations

### 1. QMIX Monotonic Hypernetwork Mixing
$$
W_1 = |W_1^{\text{raw}}(\mathbf{s})| \ge 0, \quad W_2 = |W_2^{\text{raw}}(\mathbf{s})| \ge 0
$$
$$
Q_{\text{tot}} = \text{ELU}\left(\mathbf{Q}_{\text{agents}} W_1 + \mathbf{b}_1\right) W_2 + b_2
$$

### 2. MAPPO Centralized GAE
$$
\delta_t = r_t + \gamma V(\mathbf{s}_{t+1})(1 - d_t) - V(\mathbf{s}_t)
$$
$$
\hat{A}_t^{\text{central}} = \sum_{l=0}^\infty (\gamma \lambda)^l \delta_{t+l}
$$

### 3. CFR Regret Matching
$$
\sigma^{T+1}(I, a) = \begin{cases} \frac{\max(R^T(I, a), 0)}{\sum_{a'} \max(R^T(I, a'), 0)} & \text{if } \sum_{a'} \max(R^T(I, a'), 0) > 0 \\ \frac{1}{|A(I)|} & \text{otherwise} \end{cases}
$$

### 4. Agentic Deliberation Consensus Metric
$$
S(\mathbf{e}_1, \dots, \mathbf{e}_N) = \frac{1}{N(N-1)} \sum_{i \ne j} \frac{\mathbf{e}_i \cdot \mathbf{e}_j}{\max\left(\|\mathbf{e}_i\|_2 \|\mathbf{e}_j\|_2, \, \epsilon\right)}
$$

### Asymptotic Time & Memory Complexity
- QMIX Monotonic Mixing: $\mathcal{O}(B \cdot N \cdot H)$ time and memory.
- MAPPO Centralized GAE: $\mathcal{O}(B \cdot T)$ time and memory.
- CFR Regret Matching: $\mathcal{O}(B \cdot |A|)$ time and memory.
- Agentic Reasoning Consensus: $\mathcal{O}(B \cdot N^2 \cdot D)$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def qmix_monotonic_mixing(
    agent_qs: torch.Tensor,
    w1_raw: torch.Tensor,
    b1: torch.Tensor,
    w2_raw: torch.Tensor,
    b2: torch.Tensor,
) -> torch.Tensor:
    w1 = torch.abs(w1_raw)
    w2 = torch.abs(w2_raw)
    hidden = F.elu(torch.bmm(agent_qs, w1) + b1)
    q_tot = torch.bmm(hidden, w2) + b2
    return q_tot.squeeze(-1).squeeze(-1)

def mappo_centralized_gae(
    values: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    next_value: torch.Tensor,
    gamma: float = 0.99,
    gae_lambda: float = 0.95,
) -> torch.Tensor:
    b, t = values.shape
    advantages = torch.zeros_like(values)
    gae = torch.zeros(b, dtype=values.dtype, device=values.device)
    for step in reversed(range(t)):
        next_v = next_value if step == t - 1 else values[:, step + 1]
        non_terminal = 1.0 - dones[:, step]
        delta = rewards[:, step] + gamma * next_v * non_terminal - values[:, step]
        gae = delta + gamma * gae_lambda * non_terminal * gae
        advantages[:, step] = gae
    return advantages

def cfr_regret_matching(
    cumulative_regrets: torch.Tensor,
    eps: float = 1e-9,
) -> torch.Tensor:
    positive_regrets = torch.clamp(cumulative_regrets, min=0.0)
    sum_positive = torch.sum(positive_regrets, dim=-1, keepdim=True)
    num_actions = cumulative_regrets.shape[-1]
    uniform_prob = torch.full_like(cumulative_regrets, 1.0 / num_actions)
    return torch.where(sum_positive > eps, positive_regrets / (sum_positive + eps), uniform_prob)

def agentic_consensus_metric(
    embeddings: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    b, n, d = embeddings.shape
    norms = torch.norm(embeddings, p=2, dim=-1, keepdim=True)
    normed = embeddings / torch.clamp(norms, min=eps)
    sim_matrix = torch.bmm(normed, normed.transpose(1, 2))
    diag_mask = torch.eye(n, dtype=torch.bool, device=embeddings.device).unsqueeze(0)
    off_diag_sims = sim_matrix.masked_select(~diag_mask).view(b, n * (n - 1))
    return torch.mean(off_diag_sims, dim=-1)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

q_tot = qmix_monotonic_mixing(weights["agent_qs"], weights["w1_raw"], weights["b1"], weights["w2_raw"], weights["b2"])
err_qmix = float(torch.max(torch.abs(q_tot - weights["qmix_q_tot_ref"])))
assert err_qmix < 1e-5, f"QMIX monotonic mixing parity error: {err_qmix:.2e}"

adv = mappo_centralized_gae(weights["values"], weights["rewards"], weights["dones"], weights["next_value"], gamma=0.99, gae_lambda=0.95)
err_mappo = float(torch.max(torch.abs(adv - weights["mappo_advantages_ref"])))
assert err_mappo < 1e-5, f"MAPPO centralized GAE parity error: {err_mappo:.2e}"

strat = cfr_regret_matching(weights["cumulative_regrets"], eps=1e-9)
err_cfr = float(torch.max(torch.abs(strat - weights["cfr_strategy_ref"])))
assert err_cfr < 1e-5, f"CFR regret matching parity error: {err_cfr:.2e}"

consensus = agentic_consensus_metric(weights["embeddings"], eps=1e-6)
err_consensus = float(torch.max(torch.abs(consensus - weights["consensus_ref"])))
assert err_consensus < 1e-5, f"Agentic consensus metric parity error: {err_consensus:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_qmix, err_mappo, err_cfr, err_consensus):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_qmix = benchmark_fn(
    lambda: qmix_monotonic_mixing(weights["agent_qs"], weights["w1_raw"], weights["b1"], weights["w2_raw"], weights["b2"]),
    iters=100,
    name="QMIX Monotonic Mixing",
)
b_mappo = benchmark_fn(
    lambda: mappo_centralized_gae(weights["values"], weights["rewards"], weights["dones"], weights["next_value"]),
    iters=100,
    name="MAPPO Centralized GAE",
)
b_cfr = benchmark_fn(
    lambda: cfr_regret_matching(weights["cumulative_regrets"]),
    iters=100,
    name="CFR Regret Matching",
)
b_cons = benchmark_fn(
    lambda: agentic_consensus_metric(weights["embeddings"]),
    iters=100,
    name="Agentic Consensus Metric",
)

print(format_benchmark_table([b_qmix, b_mappo, b_cfr, b_cons]))

In [ ]:
# Failure Mode Contrast: Unconstrained Linear Mixing vs QMIX Monotonic Mixing
# In unconstrained mixing, negative weights permit an agent's individual utility improvement to harm total utility
q_agent_1 = torch.tensor([[[1.0]]], requires_grad=True)
q_agent_2 = torch.tensor([[[3.0]]], requires_grad=True)

# Case 1: Unconstrained mixing with negative weight w = -2.0
w_neg = torch.tensor([[[-2.0]]])
q_tot_unconstrained_1 = torch.bmm(q_agent_1, w_neg).item()
q_tot_unconstrained_2 = torch.bmm(q_agent_2, w_neg).item()
delta_unconstrained = q_tot_unconstrained_2 - q_tot_unconstrained_1

# Case 2: QMIX monotonic mixing with enforced |w| = 2.0 >= 0
w_abs = torch.abs(w_neg)
q_tot_qmix_1 = torch.bmm(q_agent_1, w_abs).item()
q_tot_qmix_2 = torch.bmm(q_agent_2, w_abs).item()
delta_qmix = q_tot_qmix_2 - q_tot_qmix_1

print(f"Unconstrained Delta Q_tot: {delta_unconstrained:.4f} (Violates IGM: individual gain reduces team Q!)")
print(f"QMIX Monotonic Delta Q_tot: {delta_qmix:.4f} (Guarantees IGM: individual gain increases team Q!)")

assert delta_unconstrained < 0, f"Unconstrained mixing must invert gradient (got {delta_unconstrained})"
assert delta_qmix > 0, f"QMIX monotonic mixing must preserve positive correlation (got {delta_qmix})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Monotonic Hypernetwork Constraint**: Absolute value weight gating guarantees the Individual-Global-Max (IGM) condition, enabling decentralized execution without inter-agent communication.
2. **Centralized GAE Advantage**: Conditioning critics on global state $V(\mathbf{s})$ reduces policy variance dramatically without requiring state access at test time.
3. **Regret Matching Simplicity**: CFR computes game-theoretic optimal strategies via simple local positive regret normalization without linear programming solvers.
4. **Agentic Debate Convergence**: Pairwise thought cosine similarity provides an empirical stopping criterion for multi-agent LLM reasoning workflows.

---
### Cross-Domain Generalisation Matrix
- **Automated Warehouse Logistics**: QMIX coordinates fleet navigation without collision or gridlock.
- **Poker & Financial Auctions**: CFR derives unexploitable minimax strategies in imperfect information games.
- **Collaborative LLM Systems**: Multi-agent debate loops resolve complex reasoning and coding tasks through structured consensus.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.md)
- **PyTorch 2.x Module:** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.py)
- **Functional JAX Module:** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.py)
- **Native Rust Candle Track:** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.rs)
- **Hardware Kernel (CPU / WGSL):** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.rs)
- **Golden SafeTensors:** [07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Hierarchical RL Options Feudal Her and POMDP Active Inference](06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 08. Successor Representations Forward Backward Zero Shot RL and DIAYN](08_successor_representations_forward_backward_zero_shot_rl_and_diayn.ipynb) &rarr;